# TripWise RAG Demo

This notebook builds a small Retrieval-Augmented Generation (RAG) pipeline for TripWise travel data. It uses:
- sentence-transformers for embeddings
- FAISS for vector search
- a simple grounded answer generator

You can replace the sample docs with your own PostgreSQL/MySQL content or PDFs later.

In [1]:
# Install all required packages inside the notebook environment
%pip install -r rag_requirements.txt

import os
from typing import List

print('Ready to build RAG pipeline.')

  Using cached faiss_cpu-1.15.0-cp313-cp313-win_amd64.whl.metadata (7.8 kB)
  Using cached sentence_transformers-6.0.0-py3-none-any.whl.metadata (20 kB)
  Using cached openai-3.3.1-py3-none-any.whl.metadata (41 kB)
  Using cached transformers-5.15.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.28.0-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.13.0-cp313-cp313-win_amd64.whl.metadata (39 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached regex-2026.7.19-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached httpx2-2.12.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached jiter-0.16.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Usi

In [2]:
# Import libraries only after installation has completed in this kernel
import numpy as np
print('NumPy version:', np.__version__)

import faiss
print('FAISS version:', faiss.__version__)

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Embedding model loaded:', model.__class__.__name__)

NumPy version: 2.3.5
FAISS version: 1.15.0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: SentenceTransformer


## 1) Create a knowledge base

These sample documents mimic TripWise content such as destination recommendations, budgets, and trip planning guidance.

In [3]:
docs = [
    {
        'id': 'doc_1',
        'title': 'Goa travel guide',
        'text': 'Goa is best for beach holidays, nightlife, and relaxed coastal activities. For a balanced trip, visit Baga, Anjuna, and Old Goa. Budget travelers can stay in hostels or boutique guesthouses and spend 4,000 to 7,000 INR per day on food and local transport.'
    },
    {
        'id': 'doc_2',
        'title': 'Jaipur itinerary',
        'text': 'Jaipur is ideal for heritage travel and shopping. Must-visit places include Hawa Mahal, Amber Fort, and City Palace. A 3-day itinerary works well with early morning sightseeing, local markets in the afternoon, and dinner in the old city.'
    },
    {
        'id': 'doc_3',
        'title': 'Budget planning',
        'text': 'Trip budgets should include accommodation, food, local transport, attraction tickets, and emergency buffer. A practical rule is to reserve 15 percent of the total budget for contingencies and keep daily spending under control by tracking expenses in real time.'
    },
    {
        'id': 'doc_4',
        'title': 'Family trip planning',
        'text': 'Family trips do best with a slower pace. Choose 2 major cities max per week, schedule rest time, and keep group activities flexible. Early bookings help lower accommodation costs, especially during holiday seasons.'
    },
    {
        'id': 'doc_5',
        'title': 'Remote work travel',
        'text': 'For digital nomads, pick destinations with reliable Wi-Fi, cafes, and coworking spaces. Places with train connectivity and a walkable core help maintain productivity while keeping exploration easy.'
    },
]

for item in docs:
    print(item['id'], '-', item['title'])

doc_1 - Goa travel guide
doc_2 - Jaipur itinerary
doc_3 - Budget planning
doc_4 - Family trip planning
doc_5 - Remote work travel


In [4]:
def chunk_text(text: str, chunk_size: int = 150, overlap: int = 25):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk.strip())
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

chunks = []
for doc in docs:
    for chunk in chunk_text(doc['text']):
        chunks.append({
            'doc_id': doc['id'],
            'title': doc['title'],
            'text': chunk,
        })

print('Total chunks:', len(chunks))
print(chunks[0])

Total chunks: 10
{'doc_id': 'doc_1', 'title': 'Goa travel guide', 'text': 'Goa is best for beach holidays, nightlife, and relaxed coastal activities. For a balanced trip, visit Baga, Anjuna, and Old Goa. Budget travelers can'}


In [5]:
texts = [chunk['text'] for chunk in chunks]
embeddings = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype('float32'))
print('FAISS index ready with dimensions:', dim)

FAISS index ready with dimensions: 384


In [6]:
def retrieve(query: str, top_k: int = 3):
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        results.append({
            'score': float(score),
            'title': chunks[int(idx)]['title'],
            'text': chunks[int(idx)]['text'],
        })
    return results

sample_query = 'How should I plan a budget-friendly trip to Goa?'
print('Query:', sample_query)
for result in retrieve(sample_query):
    print(f"- {result['title']} | score={result['score']:.4f}")
    print(result['text'])
    print('-' * 60)

Query: How should I plan a budget-friendly trip to Goa?
- Goa travel guide | score=0.7423
Goa is best for beach holidays, nightlife, and relaxed coastal activities. For a balanced trip, visit Baga, Anjuna, and Old Goa. Budget travelers can
------------------------------------------------------------
- Budget planning | score=0.5334
Trip budgets should include accommodation, food, local transport, attraction tickets, and emergency buffer. A practical rule is to reserve 15 percent
------------------------------------------------------------
- Goa travel guide | score=0.5283
oa. Budget travelers can stay in hostels or boutique guesthouses and spend 4,000 to 7,000 INR per day on food and local transport.
------------------------------------------------------------


In [7]:
def answer_with_rag(question: str):
    context = retrieve(question, top_k=3)
    context_text = '\n\n'.join(f"[{item['title']}] {item['text']}" for item in context)
    system_prompt = (
        'You are a helpful travel planner. Answer the user using the retrieved context only. '
        'If the answer is not in the context, say clearly that the information is not available.\n'
        '\nContext:\n'
        f'{context_text}'
    )
    prompt = f"Question: {question}\n\nAnswer with practical advice in 3-5 sentences."
    final_prompt = system_prompt + '\n' + prompt

    # Fallback answer if no LLM is configured
    if not context:
        return 'I could not find relevant knowledge for this question in the current TripWise database.'

    answer = (
        'Based on the TripWise knowledge base, ' + context[0]['text'] + ' ' +
        'This is relevant to your question because it covers the most similar travel guidance in the available documents.'
    )
    return answer

question = 'What is the best way to plan a family trip without overspending?'
print(answer_with_rag(question))

Based on the TripWise knowledge base, Family trips do best with a slower pace. Choose 2 major cities max per week, schedule rest time, and keep group activities flexible. Early bookings he This is relevant to your question because it covers the most similar travel guidance in the available documents.


In [8]:
while True:
    user_question = input('Ask a travel question (or type exit): ').strip()
    if user_question.lower() in {'exit', 'quit'}:
        print('Goodbye!')
        break
    print('---')
    print(answer_with_rag(user_question))
    print('---')

---
Based on the TripWise knowledge base, exible. Early bookings help lower accommodation costs, especially during holiday seasons. This is relevant to your question because it covers the most similar travel guidance in the available documents.
---
---
Based on the TripWise knowledge base, walkable core help maintain productivity while keeping exploration easy. This is relevant to your question because it covers the most similar travel guidance in the available documents.
---
Goodbye!
